# Offline Validation + MLflow Lifecycle

Validates the **deployed champion models** (XGBoost, Isolation Forest, LSTM) against
held-out labeled data and logs metrics to **MLflow** for champion/challenger promotion.

This notebook is **off the real-time path** — it never runs during `/evaluate` or Kafka
orchestration. Run it weekly after retraining or before promoting a new model artifact.

**Metrics:** PR-AUC (primary for 1.8% fraud rate), AUROC, recall@P≥20%, best F1.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

# Locate backend/ from notebooks/ or repo root
base = Path.cwd()
while not (base / "eval" / "offline_validation.py").exists():
    if base.parent == base:
        raise FileNotFoundError("Run from backend/ or backend/notebooks/")
    base = base.parent
if str(base) not in sys.path:
    sys.path.insert(0, str(base))

from eval.offline_validation import run_all, validate_xgboost, validate_isolation_forest, validate_lstm
from eval.mlflow_tracking import configure_mlflow, champion_challenger_decision
from eval.paths import MLRUNS_DIR

print("backend:", base)
print("mlruns:", MLRUNS_DIR)

## 1. Validate each model (no MLflow)

In [ ]:
xgb = validate_xgboost(log_mlflow=False)
iso = validate_isolation_forest(log_mlflow=False)
lstm = validate_lstm(log_mlflow=False)

rows = []
for name, r in [("xgboost", xgb), ("isolation_forest", iso), ("lstm", lstm)]:
    m = r["metrics"]
    rows.append({
        "model": name,
        "n": m.get("n"),
        "pr_auc": round(m.get("pr_auc", float("nan")), 4),
        "auroc": round(m.get("auroc", float("nan")), 4),
        "recall_at_p20": round(m.get("recall_at_p20", float("nan")), 4),
        "best_f1": round(m.get("best_f1", float("nan")), 4),
    })
summary_df = pd.DataFrame(rows)
summary_df

## 2. Log to MLflow + champion/challenger gate

Tracking URI: `file:backend/mlruns/` (local). View UI:

```bash
cd backend && uv run mlflow ui --port 5000
```

In [ ]:
uri = configure_mlflow()
print("MLflow tracking URI:", uri)

report = run_all(log_mlflow=True)
print(json.dumps(report["summary"], indent=2))

for model, r in report["results"].items():
    gate = r.get("promotion_gate")
    if gate:
        print(f"{model}: {gate['decision'].upper()} — {gate['reason']}")

## 3. Compare challenger vs rule-engine baseline

In [ ]:
from eval.offline_validation import validate_rule_baseline

baseline = validate_rule_baseline(log_mlflow=False)
xgb_pr = report["results"]["xgboost"]["metrics"]["pr_auc"]
print("XGBoost PR-AUC:", round(xgb_pr, 4))
print("Rule baseline recall:", round(baseline["metrics"]["recall"], 4))
print("Rule baseline precision:", round(baseline["metrics"]["precision"], 4))